# A line in a bigger space

Some labels are not separable by a straight line. The practical move is to lift each point into extra coordinates, then run the same SVM there.

Take two coordinates $(x_1, x_2)$ and build

$$
\varphi(x_1, x_2) = (x_1^2,\ \sqrt{2}\, x_1 x_2,\ x_2^2)
$$

The $\sqrt{2}$ is there so the dot product becomes a square of the original dot product. Expand it:

$$
\begin{aligned}
\varphi(\mathbf{x})\cdot\varphi(\mathbf{z})
&= x_1^2 z_1^2 + 2 x_1 x_2 z_1 z_2 + x_2^2 z_2^2 \\
&= (x_1 z_1 + x_2 z_2)^2 \\
&= (\mathbf{x}\cdot\mathbf{z})^2
\end{aligned}
$$

The right-hand side is the degree-$2$ polynomial kernel. You can compute it from $\mathbf{x}$ and $\mathbf{z}$ without storing $\varphi$.


## One numerical expansion

Let $\mathbf{x} = (1, 2)$ and $\mathbf{z} = (3, -1)$.

$$
\mathbf{x}\cdot\mathbf{z} = 3 - 2 = 1, \qquad (\mathbf{x}\cdot\mathbf{z})^2 = 1
$$

The lifted vectors are

$$
\begin{aligned}
\varphi(\mathbf{x}) &= (1,\ 2\sqrt{2},\ 4) \\
\varphi(\mathbf{z}) &= (9,\ -3\sqrt{2},\ 1)
\end{aligned}
$$

$$
\varphi(\mathbf{x})\cdot\varphi(\mathbf{z})
= 9 + (2\sqrt{2})(-3\sqrt{2}) + 4
= 9 - 12 + 4
= 1
$$

Same number, two routes. The kernel route never built a square root.


In [1]:
# Both routes for (x·z)^2. The square roots cancel in the dot product.
import math

x = (1, 2)
z = (3, -1)
kernel = (x[0] * z[0] + x[1] * z[1]) ** 2
phi_x = (x[0] ** 2, math.sqrt(2) * x[0] * x[1], x[1] ** 2)
phi_z = (z[0] ** 2, math.sqrt(2) * z[0] * z[1], z[1] ** 2)
lifted = sum(a * b for a, b in zip(phi_x, phi_z))
assert kernel == 1
assert abs(lifted - kernel) < 1e-12
print("kernel", kernel)
print("lifted dot product", lifted)


kernel 1
lifted dot product 0.9999999999999982


## A different map is a different kernel

The product map

$$
\psi(x_1, x_2) = (x_1,\ x_2,\ x_1 x_2)
$$

is useful, and it is not the map above. Its dot product is

$$
\psi(\mathbf{x})\cdot\psi(\mathbf{z})
= \mathbf{x}\cdot\mathbf{z} + (x_1 x_2)(z_1 z_2)
$$

That is not $(\mathbf{x}\cdot\mathbf{z})^2$. If a lesson claims a kernel, the feature map has to match it term by term. A later lesson uses $\psi$ on the XOR pattern because the product coordinate separates that pattern with integer arithmetic. The $\sqrt{2}$ map remains the one that matches $(\mathbf{x}\cdot\mathbf{z})^2$.


In [2]:
# The product map and the squared-dot-product kernel disagree on these vectors.
x = (1, 2)
z = (3, -1)
squared_dot = (x[0] * z[0] + x[1] * z[1]) ** 2
product_kernel = (x[0] * z[0] + x[1] * z[1]) + (x[0] * x[1]) * (z[0] * z[1])
assert squared_dot == 1
assert product_kernel == 1 + (2) * (-3)
assert product_kernel != squared_dot
print("squared dot product", squared_dot)
print("product-map kernel", product_kernel)


squared dot product 1
product-map kernel -5


## A pitfall

Adding coordinates can separate training points and still behave badly on new points. The kernel is not a free lunch. It is a choice of which comparisons, such as products or squared distances, the street is allowed to use.

## What to carry forward

$(\mathbf{x}\cdot\mathbf{z})^2$ equals the dot product of $(x_1^2,\ \sqrt{2} x_1 x_2,\ x_2^2)$ with the same map at $\mathbf{z}$. A product coordinate $x_1 x_2$ belongs to a different kernel.
